# Modül 07: Transfer Learning
## Deep Learning Path
---
## İçindekiler
1. Öğrenme Hedefleri
2. Neden Transfer Learning?
3. 3 Strateji
4. NumPy Implementasyon
5. TF/PyTorch
6. Diferansiyel LR
7. Alıştırmalar

## 1. Öğrenme Hedefleri
- Özellik hiyerarşisini açıklamak
- Feature extraction vs fine-tuning seçmek
- Freeze/unfreeze uygulamak
- Diferansiyel LR kullanmak
- TF ve PyTorch'ta pretrained model kullanmak

## 2. Neden Transfer Learning?

```
Katman 1-3  : Kenarlar, köşeler  → %95 transfer edilebilir
Katman 4-8  : Parçalar, şekiller → %75 transfer edilebilir
Katman 9-12 : Nesneler           → %40 transfer edilebilir
FC Katman   : Sınıflandırıcı    → Her zaman yeniden eğit
```

ImageNet öğrenimleri çoğu görüntü görevine transfer edilir.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

layers = ['Katman 1-3\n(Kenarlar)', 'Katman 4-8\n(Şekiller)',
          'Katman 9-12\n(Nesneler)', 'FC\n(Sınıflandırıcı)']
transfer = [0.95, 0.75, 0.40, 0.05]
colors = ['#2E7D32','#1565C0','#E65100','#B71C1C']
fig,ax = plt.subplots(figsize=(9,3.5))
bars = ax.barh(layers, transfer, color=colors, edgecolor='white', lw=1.5)
for bar,val in zip(bars,transfer):
    ax.text(val+.02, bar.get_y()+bar.get_height()/2, f'{val:.0%}', va='center', fontweight='bold')
ax.set_xlim(0,1.15); ax.set_xlabel('Transfer Edilebilirlik')
ax.set_title('CNN Özellik Hiyerarşisi', fontweight='bold')
ax.axvline(.5,color='gray',ls='--',alpha=.5); ax.grid(True,alpha=.3,axis='x')
plt.tight_layout(); plt.show()


## 3. Strateji Seçim Rehberi

| Veri | Domain | Strateji | LR |
|------|--------|----------|----|
| Az (<1K) | Benzer | Feature Extraction | 1e-3 |
| Orta | Benzer | Fine-tune (son bloklar) | 1e-4 |
| Orta | Farklı | Fine-tune (derin) | 1e-4 |
| Çok (>100K) | Farklı | Full Training | 1e-3 |

In [ ]:
def relu(x): return np.maximum(0,x)

class Layer:
    def __init__(self,ni,no,frozen):
        self.W=np.random.randn(ni,no)*.1; self.b=np.zeros(no)
        self.frozen=frozen; self.dW=None
    def forward(self,x): self.x=x; return relu(x@self.W+self.b)
    def backward(self,dout):
        if self.frozen: return dout@self.W.T
        self.dW=self.x.T@dout; return dout@self.W.T
    def update(self,lr):
        if not self.frozen and self.dW is not None: self.W-=lr*self.dW

class TransferModel:
    def __init__(self,strategy):
        fm = {'feature_extract':[True,True],'fine_tune':[True,False],'full':[False,False]}
        self.layers=[Layer(32,64,fm[strategy][0]),Layer(64,128,fm[strategy][1])]
        self.Wh=np.random.randn(128,5)*.1; self.bh=np.zeros(5)
    def softmax(self,x):
        e=np.exp(x-x.max(1,keepdims=True)); return e/e.sum(1,keepdims=True)
    def forward(self,x):
        for l in self.layers: x=l.forward(x)
        self.feat=x; return self.softmax(x@self.Wh+self.bh)
    def step(self,x,y,lr=.005):
        p=self.forward(x); n=y.shape[0]
        loss=-np.mean(np.sum(y*np.log(p+1e-15),1))
        dlogits=(p-y)/n; dWh=self.feat.T@dlogits
        self.Wh-=lr*dWh; self.bh-=lr*dlogits.sum(0)
        dout=dlogits@self.Wh.T
        for l in reversed(self.layers): dout=l.backward(dout); l.update(lr)
        return loss

np.random.seed(0)
X=np.random.randn(100,32); y=np.eye(5)[np.random.randint(0,5,100)]
fig,ax=plt.subplots(figsize=(10,4))
for strategy,color in [('feature_extract','#1565C0'),('fine_tune','#00695C'),('full','#E65100')]:
    m=TransferModel(strategy)
    hist=[m.step(X,y) for _ in range(80)]
    trainable=sum(l.W.size for l in m.layers if not l.frozen)+m.Wh.size
    ax.plot(hist,color=color,lw=2,label=f'{strategy} ({trainable:,} param)')
ax.set_title('3 Strateji — Loss Yakınsama',fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(True,alpha=.3)
plt.tight_layout(); plt.show()


## 4. Diferansiyel Learning Rate

Fine-tuning'de backbone için küçük, yeni katmanlar için büyük lr kullan.

In [ ]:
# PyTorch diferansiyel LR konsepti
diff_lr_code = '''
import torchvision.models as models
model = models.resnet50(pretrained=True)

optimizer = torch.optim.Adam([
    {"params": model.layer1.parameters(), "lr": 1e-5},  # cok dusuk
    {"params": model.layer2.parameters(), "lr": 1e-5},
    {"params": model.layer3.parameters(), "lr": 1e-4},  # orta
    {"params": model.layer4.parameters(), "lr": 1e-4},
    {"params": model.fc.parameters(),     "lr": 1e-3},  # yuksek - yeni katman
])
'''
print(diff_lr_code)
print("Kural: Alt katmanlar kucuk lr → iyi ozellikler korunsun")
print("       Ust katmanlar + head buyuk lr → yeni goreve uyum")


## 5. Alıştırmalar

**1.** Feature extraction vs fine-tuning'i 10 ve 100 örnekle karşılaştır.

**2.** PyTorch'ta ResNet50'yi yükle, `.fc` katmanını değiştir, parametre sayısını hesapla.

**3.** Layer-wise LR decay uygula: `lr = base_lr * 0.9^depth`.

**4.** X-ray görüntüleri için hangi strateji? Gerekçelendir.

**5.** MobileNetV2'yi feature extractor olarak kullan, 5 sınıf üzerinde eğit.

---
**Sonraki Modül:** Modül 08 — RNN / LSTM / GRU